# Lesson 3: Classes

Enough object-oriented programming to understand every tool you import.

<img src="../img/map-3.svg" width="85%" alt="">

## How to follow along, and Quizlets

You should have the notebooks with the name `lecture-workbook` open, **not these slides**. Whenever you see a quizlet, you should do the exercise in your notebook and then we will go over the answer together. You will have a a few minutes for each exercise. 

## Why classes

You have been using classes all along: every array, histogram and figure was defined as a class by some library.

In project 2 you already leaned on a class: `vector`'s arrays bundle the data with `.mass` and `+`. A class keeps data and behaviour together and gives the combination a name. Today you see what that takes.

## Objects

An object is a collection of data and the functions that operate on that data.

Everything in Python is one. A function attached to an object is called a *method*, and a dot calls it.

In [1]:
import numpy as np

pt = np.array([63.0, 38.1, 4.0])
pt.max(), pt.shape, "muon".upper()

(np.float64(63.0), (3,), 'MUON')

## Classes

A class is the template for one kind of object: which data each object carries, and which methods work on it.
You call the class like a function, and it hands you a new object made from the template.

<img src="../img/class-objects.svg" width="85%" alt="">

## Everything you used was a class

`type` from lesson 1 names the class that made each value. Arrays, histograms and even floats are objects of classes that someone defined.

In [2]:
import hist

type(pt), type(63.0), type(hist.Hist.new.Reg(10, 0, 1).Double())

(numpy.ndarray, float, hist.hist.Hist)

## Defining your own class

`class` opens the definition. `__init__` says what every new object starts with.

What happens, step by step, when you call a class:

1. Python creates a new, empty object.
2. It calls the class's `__init__` method, passing that empty object in as `self`, followed by your arguments.
3. Each `self.something = value` line attaches one piece of data to the object.
4. The finished object comes back, for you to name.

In [3]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

mu = Muon(63.0, 1)
mu.pt

63.0

The object carries exactly what `__init__` attached to it, and nothing else:

In [4]:
mu.__dict__

{'pt': 63.0, 'charge': 1}

## Separate data

Each object keeps its own data.

**Quizlet 3.1 — separate data.** Before you run it: why is the result not `38.1, 38.1`?

In [5]:
other = Muon(38.1, -1)
mu.pt, other.pt

(63.0, 38.1)

## Methods

A method is a function defined inside the class. Its first argument is always `self`.

In [6]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

    def opposite(self, other):
        return self.charge != other.charge

Muon(63.0, 1).opposite(Muon(38.1, -1))

True

## Printing your objects

Without help, Python prints `<__main__.Muon object at 0x10...>`. `__repr__` returns the text to show instead. Write it once, read it everywhere.

In [7]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

    def __repr__(self):
        return f"Muon(pt={self.pt}, charge={self.charge})"


Muon(63.0, 1)

Muon(pt=63.0, charge=1)

## Quizlet 3.2 — who passed the argument?

```python
class Muon:
    def flipped():
        return ...

mu.flipped()
```

`TypeError: flipped() takes 0 positional arguments but 1 was given.`

You passed nothing. Who passed the argument?

<img src="../img/self-call.svg" width="75%" alt="">


Python did. A method call on an object is shorthand: `mu.flipped()` is exactly `Muon.flipped(mu)`, and you can run that second form yourself. Python passes the object in as the first argument automatically; `self` is the parameter that receives it — and `def flipped():` left no slot for it.

In [8]:
"mu".upper(), str.upper("mu")

('MU', 'MU')

## Properties

`@property` makes a method read like data: `z.mass`, with no brackets.

This is how `vector` gives you `.mass` in lesson 4.

In [9]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

FourVector(5.0, 1.0, 2.0, 2.0).mass

np.float64(4.0)

**Quizlet 3.3 — property or method?** One of `z.mass` and `z.mass()` works. Which one, and what error does the other give?

`z.mass` — a property reads like data. `z.mass()` tries to call the *number* it returned: `TypeError: 'float' object is not callable`.

## Quizlet 3.4 — assigning to a property

`z.mass` is computed from the four components. What happens on

```python
z.mass = 999.0
```

Guess before you run.

In [10]:
z = FourVector(91.2, 0.0, 0.0, 0.0)
z.mass = 999.0

AttributeError: property 'mass' of 'FourVector' object has no setter

A property without a setter cannot be assigned. Your objects can refuse nonsense: nobody can hand this analysis an inconsistent four-vector.

## The `@` sign is a decorator

A decorator is a function that is applied to the function defined under it.
`@property` hands your method to `property()`, which makes it read like data. You do not need to write decorators to use them — but now the syntax is not magic.

Defining one is ordinary Python — a function that takes a function and returns a function. `@logged` is nothing but shorthand for `square = logged(square)`:

In [ ]:
def logged(function):
    def wrapped(x):
        print("called with", x)
        return function(x)
    return wrapped


@logged
def square(x):
    return x**2


square(4)

## `__add__`

`__add__` tells Python what `+` means for your objects.

Names with two underscores on each side are special methods. Python calls them for you.

In [23]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    def __add__(self, other):
        return FourVector(
            self.E + other.E,
            self.px + other.px,
            self.py + other.py,
            self.pz + other.pz,
        )

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

(FourVector(5.0, 1.0, 2.0, 2.0) + FourVector(5.0, -1.0, -2.0, -2.0)).mass

np.float64(10.0)

## Operators are method calls

`a + b` runs `a.__add__(b)`. `len(x)` runs `x.__len__()`. `a > b` runs `a.__gt__(b)`. The double underscores mark the methods that Python itself calls.

That is the whole trick behind lesson 2: NumPy wrote `__gt__` to return a mask.

In [24]:
pt.__gt__(20.0)

array([ True,  True, False])

`==` is `__eq__`. Without it, `==` means "the very same object":

In [25]:
Muon(63.0, 1) == Muon(63.0, 1)

True

Two equal muons compare as `False` until you define `__eq__`. Every operator is yours to define, and only the ones you define exist.

## Why you can chain dots

Lesson 2 built histograms with `hist.Hist.new.Reg(50, 0, 100).Double()`. Each piece returns an object, and the next dot asks that object. A chain is nothing but "ask the result again".

**Quizlet 3.5 — chains.** In `"muon".upper().count("M")`, what does each link hand to the next? Check with `type`:

In [26]:
type("muon".upper()), "muon".upper().count("M")

(hist.quick_construct.QuickConstruct, hist.quick_construct.ConstructProxy)

`.upper()` returns a new string, so anything a string can do can follow it. `hist.Hist.new.Reg(50, 0, 100).Double()` from lesson 2 works the same way: every link returns an object, and the next dot asks that object.

## Columns of particles

The same class works for one particle or for a whole column of particles.

If the data are NumPy arrays, every method works element by element.

In [15]:
E = np.array([5.0, 10.0])
px = np.array([1.0, 6.0])
zeros = np.zeros(2)
FourVector(E, px, zeros, zeros).mass

array([4.89897949, 8.        ])

## Settings

A class can hold settings that its methods use.

In [16]:
class Selection:
    def __init__(self, min_pt):
        self.min_pt = min_pt

    def apply(self, pt):
        return pt[pt > self.min_pt]

Selection(min_pt=20).apply(pt)

array([63. , 38.1])

## Inheritance

A new class starts from an existing one and changes only what differs.

`super().__init__(...)` runs the parent's `__init__`.

**Quizlet 3.6 — inheritance.** `TightSelection` defines no `apply` of its own. What does the call give, and whose code runs?

In [17]:
class TightSelection(Selection):
    def __init__(self):
        super().__init__(min_pt=50)

TightSelection().apply(pt)

array([63.])

## Catching errors

Lesson 1 taught you to *read* errors. `try`/`except` lets your code *survive* one you expect, and only the one you name — anything else should still crash loudly.

In [18]:
event = {"electron": []}

try:
    first = event["muon"][0]
except KeyError:
    first = None

print(first)

None


## `isinstance`

An object of a child class still counts as the parent: code written for `Selection` accepts a `TightSelection` without knowing it exists. That is the point of inheritance.

In [19]:
isinstance(TightSelection(), Selection)

True

## Results as dictionaries

An analysis of one piece of data fits in a dictionary. Two results combine key by key, with the one-line dictionary form from lesson 1.

<img src="../img/two-ways.svg" width="80%" alt="">

In [20]:
first = {"events": 3, "passed": 2}
second = {"events": 2, "passed": 1}

total = {key: first[key] + second[key] for key in first}
total

{'events': 5, 'passed': 3}

## In real code: `@dataclass`

When a class is mostly stored data, Python writes `__init__` and `__repr__` for you. You wrote them by hand today so that nothing in them is magic.

In [21]:
from dataclasses import dataclass

@dataclass
class Muon:
    pt: float
    charge: int


Muon(63.0, 1)

Muon(pt=63.0, charge=1)

`pt: float` is a *type hint*: documentation that tools can check. Python itself does not enforce it.

`@dataclass` also writes `__eq__` for you — the two equal muons that compared as `False` earlier now compare as `True`.

In [22]:
Muon(63.0, 1) == Muon(63.0, 1)

True

## Now: project 3.

[project-higgs.ipynb](project-higgs.ipynb): a four-vector class that runs the whole analysis.

<img src="../img/class-objects.svg" width="70%" alt="">